<a href="https://colab.research.google.com/github/mchlkan/EoDA_Group_Project/blob/main/Projects/CPU_Baseline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# CPU Baseline — CFPB Consumer Complaints (Machine Learning Assignment)

**Course:** Engineering of Data Analysis
**Runtime:** Colab CPU runtime (no GPU required)

**Purpose**

This notebook establishes the **CPU reference timings** against which `GPU_Solution.ipynb`
is compared. It is one half of a two-notebook benchmark; both notebooks execute the same
pipeline (feature engineering → Logistic Regression → XGBoost) on identical data splits
and with the same hyperparameters — the only difference is the execution engine.

**Pipeline**

The underlying task is binary classification of CFPB consumer complaints (`Consumer disputed?`,
Yes/No) using the same feature engineering and models as the original ML assignment
(`72782_Complaints_Notebook.ipynb`): target-encoded categoricals, narrative-derived text features,
temporal and response-quality flags, fed into
- (1) an L2-regularised Logistic Regression and
- (2) an XGBoost classifier.

**Benchmark design**

1. **Hyperparameter search (context timing).** One full cross-validated search per model on
   the complete training set. Best parameters are persisted to `results/best_params.json`
   and reused by the GPU notebook — this guarantees that the CPU and GPU timings are measuring
   the *same* fitted model, not two different models that each found their own optimum.

2. **Dataset-size sweep.** 3 runs × 5 fractions (10%, 25%, 50%, 75%, 100%) × 6 phases
   (`fe_train`, `fe_test`, `lr_fit`, `lr_predict`, `xgb_fit`, `xgb_predict`) = 90 timing rows,
   written to `results/timings_cpu.csv`. Phase-level granularity lets the downstream analysis
   report training vs. inference speedups separately, as required by the assignment brief.

**Outputs**

- `results/best_params.json` — best hyperparameters + CV context timings (consumed by GPU notebook)
- `results/timings_cpu.csv` — 90-row long-format timing log (consumed by analysis notebook)

## Table of Contents

1. [Setup & Imports](#1-setup--imports)
2. [Data Loading & Train/Test Split](#2-data-loading--traintest-split)
3. [Feature Engineering](#3-feature-engineering)
4. [Hyperparameter Search (Context Timing)](#4-hyperparameter-search-context-timing)
    - 4.1 [Logistic Regression — GridSearchCV](#41-logistic-regression--gridsearchcv)
    - 4.2 [XGBoost — RandomizedSearchCV](#42-xgboost--randomizedsearchcv)
    - 4.3 [Persist Best Parameters](#43-persist-best-parameters)
5. [Dataset-Size Sweep](#5-dataset-size-sweep)
    - 5.1 [Timing Utility](#51-timing-utility)
    - 5.2 [Sweep Execution](#52-sweep-execution)
    - 5.3 [Sanity Check](#53-sanity-check)

## 1. Setup & Imports

In [1]:
# importing necessary libraries

import numpy as np
import pandas as pd
import time
import json
import os

from sklearn.model_selection import train_test_split, GridSearchCV, RandomizedSearchCV, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from xgboost import XGBClassifier

import warnings
warnings.filterwarnings('ignore')

# fixing random seed for reproducability

np.random.seed(42)


In [ ]:
# Adjust path if file is in Google Drive
# from google.colab import drive; drive.mount('/content/drive')
# DATA_PATH = '/content/drive/MyDrive/complaints_training.csv'
DATA_PATH = 'complaints_training.csv'

df_raw = pd.read_csv(DATA_PATH)

X = df_raw.drop(columns=['Complaint ID', 'Consumer disputed?'])
y = df_raw['Consumer disputed?'].map({'Yes': 1, 'No': 0})

X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=42
)

print(f'Loaded {len(df_raw):,} rows. Train: {len(X_train_raw):,}  Test: {len(X_test_raw):,}')
print(f'Train positive rate: {y_train.mean():.3f}')


In [ ]:
def feature_engineering(X, y=None, fit_encoders=None):
    encoders = {} if fit_encoders is None else fit_encoders
    training = fit_encoders is None
    raw = X.copy()
    for col in ['Complaint ID', 'Consumer disputed?']:
        if col in raw.columns:
            raw = raw.drop(columns=[col])
    out = pd.DataFrame(index=raw.index)

    date_rec  = pd.to_datetime(raw['Date received'],        errors='coerce')
    date_sent = pd.to_datetime(raw['Date sent to company'], errors='coerce')
    out['response_lag_days'] = (date_sent - date_rec).dt.days.clip(0, 120).fillna(0).astype(int)
    out['received_year']     = date_rec.dt.year.fillna(2015).astype(int)
    out['day_of_week']       = date_rec.dt.dayofweek.fillna(0).astype(int)

    resp = raw['Company response to consumer'].fillna('Missing')
    out['company_response_monetary']       = resp.str.contains('monetary',       case=False).astype(int)
    out['company_response_non_monetary']   = resp.str.contains('non-monetary',   case=False).astype(int)
    out['company_response_explanation']    = resp.str.contains('explanation',    case=False).astype(int)
    out['company_response_without_relief'] = resp.str.contains('without relief', case=False).astype(int)
    out['company_response_in_progress']    = resp.str.contains('in progress',    case=False).astype(int)
    out['company_response_is_relief']      = (
        (out['company_response_monetary'] == 1) |
        (out['company_response_non_monetary'] == 1)
    ).astype(int)
    out['timely_response']     = raw['Timely response?'].fillna('No').eq('Yes').astype(int)
    out['closed_x_timely']     = out['company_response_is_relief'] * out['timely_response']
    out['has_public_response'] = raw['Company public response'].notna().astype(int)

    narr = raw['Consumer complaint narrative'].fillna('')
    out['has_narrative']       = (narr != '').astype(int)
    out['word_count']          = narr.apply(lambda x: len(x.split()) if x else 0)
    out['char_count']          = narr.str.len().fillna(0).astype(int)
    out['avg_word_length']     = narr.apply(
        lambda x: np.mean([len(w) for w in x.split()]) if x.split() else 0
    )
    out['narrative_short']     = (out['word_count'].between(1, 19)).astype(int)
    out['narrative_long']      = (out['word_count'] > 200).astype(int)
    esc = 'attorney|lawyer|lawsuit|legal action|sue|court|cfpb|regulation|violation|fraud|illegal|discriminat'
    out['escalation_term_count'] = narr.str.lower().str.count(esc).fillna(0).astype(int)
    out['has_escalation_terms']  = (out['escalation_term_count'] > 0).astype(int)
    out['narrative_x_no_relief'] = out['has_narrative'] * (1 - out['company_response_is_relief'])

    consent = raw['Consumer consent provided?'].fillna('Missing')
    out['consent_provided']  = consent.eq('Consent provided').astype(int)
    out['consent_withdrawn'] = consent.eq('Consent withdrawn').astype(int)
    out['has_subproduct']    = raw['Sub-product'].notna().astype(int)
    out['has_subissue']      = raw['Sub-issue'].notna().astype(int)
    tags = raw['Tags'].fillna('')
    out['tags_any']          = (tags != '').astype(int)
    out['is_older_american'] = tags.str.contains('Older American', case=False).astype(int)
    out['is_servicemember']  = tags.str.contains('Servicemember',  case=False).astype(int)

    def target_encode(series, name, smoothing=20):
        if training:
            tmp = pd.DataFrame({'val': series.values, 'target': y.values}, index=series.index)
            agg = tmp.groupby('val')['target'].agg(['mean', 'count'])
            global_mean = y.mean()
            smooth = (agg['count'] * agg['mean'] + smoothing * global_mean) / (agg['count'] + smoothing)
            encoders[name] = {'map': smooth.to_dict(), 'global_mean': global_mean}
        return series.map(encoders[name]['map']).fillna(encoders[name]['global_mean'])

    def freq_encode(series, name):
        if training:
            encoders[name] = series.value_counts().to_dict()
        return series.map(encoders[name]).fillna(1)

    out['Product_te']           = target_encode(raw['Product'].fillna('Missing'),                      'Product_te')
    out['Issue_te']             = target_encode(raw['Issue'].fillna('Missing'),                        'Issue_te')
    out['State_te']             = target_encode(raw['State'].fillna('Missing'),                        'State_te')
    out['Company_te']           = target_encode(raw['Company'].fillna('Missing'),                      'Company_te')
    out['Sub_product_te']       = target_encode(raw['Sub-product'].fillna('Missing'),                  'Sub_product_te')
    out['Sub_issue_te']         = target_encode(raw['Sub-issue'].fillna('Missing'),                    'Sub_issue_te')
    out['Submitted_via_te']     = target_encode(raw['Submitted via'].fillna('Missing'),                'Submitted_via_te')
    out['company_response_te']  = target_encode(raw['Company response to consumer'].fillna('Missing'), 'company_response_te')
    out['zip3_region_te']       = target_encode(
        raw['ZIP code'].fillna('000').astype(str).str[:3],                                             'zip3_region_te')
    prod_x_channel = raw['Product'].fillna('Missing') + '_' + raw['Submitted via'].fillna('Missing')
    out['product_x_channel_te'] = target_encode(prod_x_channel, 'product_x_channel_te')
    prod_x_issue = raw['Product'].fillna('Missing') + '_' + raw['Issue'].fillna('Missing')
    out['product_issue_te']     = target_encode(prod_x_issue, 'product_issue_te')
    out['company_volume']       = freq_encode(raw['Company'].fillna('Missing'), 'company_volume')
    out['response_te_x_relief'] = out['company_response_te'] * out['company_response_is_relief']
    return out, encoders


In [ ]:
# Feature-engineer the full training set once — used for CV only.
X_train_enc, encoders = feature_engineering(X_train_raw, y=y_train)
X_test_enc, _         = feature_engineering(X_test_raw, fit_encoders=encoders)

print(f'Features: {X_train_enc.shape[1]}  Nulls in train: {X_train_enc.isnull().sum().sum()}')


In [ ]:
# LR GridSearchCV — run ONCE on the full training set.
# Records best hyperparams and a context timing (not part of the sweep).
lr_pipeline = Pipeline([
    ('imputer',    SimpleImputer(strategy='median')),
    ('scaler',     StandardScaler()),
    ('classifier', LogisticRegression(random_state=42, max_iter=1000))
])

param_grid_lr = {
    'classifier__C':            [0.001, 0.01, 0.1, 1, 10],
    'classifier__penalty':      ['l1', 'l2'],
    'classifier__class_weight': ['balanced'],
    'classifier__solver':       ['liblinear']
}

cv_lr = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
search_lr = GridSearchCV(lr_pipeline, param_grid_lr, cv=cv_lr, scoring='f1', n_jobs=1, verbose=1)

t0 = time.perf_counter()
search_lr.fit(X_train_enc, y_train)
t_lr_cv = time.perf_counter() - t0

y_prob_lr = search_lr.best_estimator_.predict_proba(X_test_enc)[:, 1]
auc_lr    = roc_auc_score(y_test, y_prob_lr)

print(f'Best LR params: {search_lr.best_params_}')
print(f'CV F1:          {search_lr.best_score_:.4f}')
print(f'Test ROC-AUC:   {auc_lr:.4f}')
print(f'[CONTEXT] LR GridSearchCV (50 fits): {t_lr_cv:.1f}s  ({t_lr_cv/60:.1f} min)')


In [ ]:
# XGB RandomizedSearchCV — run ONCE on the full training set.
xgb_pipeline = Pipeline([
    ('imputer',    SimpleImputer(strategy='median')),
    ('classifier', XGBClassifier(
        random_state=42, eval_metric='logloss',
        tree_method='hist', device='cpu'
    ))
])

param_grid_xgb = {
    'classifier__n_estimators':     [400, 500, 600, 800],
    'classifier__max_depth':        [4, 5, 6],
    'classifier__learning_rate':    [0.02, 0.03, 0.05],
    'classifier__scale_pos_weight': [3.5, 4, 4.5],
    'classifier__min_child_weight': [5, 10, 15],
    'classifier__subsample':        [0.55, 0.6, 0.65, 0.7],
    'classifier__colsample_bytree': [0.6, 0.65, 0.7],
    'classifier__reg_alpha':        [0, 0.1, 1],
    'classifier__reg_lambda':       [3, 5, 7]
}

cv_xgb = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
search_xgb = RandomizedSearchCV(
    xgb_pipeline, param_grid_xgb,
    n_iter=60, cv=cv_xgb, scoring='f1',
    random_state=42, n_jobs=1, verbose=1
)

t0 = time.perf_counter()
search_xgb.fit(X_train_enc, y_train)
t_xgb_cv = time.perf_counter() - t0

y_prob_xgb = search_xgb.best_estimator_.predict_proba(X_test_enc)[:, 1]
auc_xgb    = roc_auc_score(y_test, y_prob_xgb)

print(f'Best XGB params: {search_xgb.best_params_}')
print(f'CV F1:           {search_xgb.best_score_:.4f}')
print(f'Test ROC-AUC:    {auc_xgb:.4f}')
print(f'[CONTEXT] XGB RandomizedCV (300 fits): {t_xgb_cv:.1f}s  ({t_xgb_cv/60:.1f} min)')


In [ ]:
# Save best hyperparams and context timings for use by GPU_Solution.ipynb
os.makedirs('results', exist_ok=True)

best_lr_params  = {k.replace('classifier__', ''): v for k, v in search_lr.best_params_.items()}
best_xgb_params = {k.replace('classifier__', ''): v for k, v in search_xgb.best_params_.items()}

best_params = {
    'lr':  best_lr_params,
    'xgb': best_xgb_params,
    'auc': {'lr': float(auc_lr), 'xgb': float(auc_xgb)},
    'cv_context_seconds': {
        'lr_gridsearch':        t_lr_cv,
        'xgb_randomizedsearch': t_xgb_cv
    }
}

with open('results/best_params.json', 'w') as f:
    json.dump(best_params, f, indent=2)

print('Saved results/best_params.json')
print(json.dumps(best_params, indent=2))


In [ ]:
# Timing utility — every timed call goes through log_row to guarantee schema consistency.
# Columns: device, fraction, n_rows, phase, run_idx, seconds, notes

_rows = []

def log_row(device, fraction, n_rows, phase, run_idx, seconds, notes=''):
    _rows.append({
        'device': device, 'fraction': fraction, 'n_rows': n_rows,
        'phase': phase, 'run_idx': run_idx, 'seconds': seconds, 'notes': notes
    })


In [ ]:
# Dataset-size sweep: 3 runs × 5 fractions × 6 phases = 90 rows.
# Sampling: df.sample(frac=..., random_state=42) — same seed = nested subsets.
# Best params from CV are fixed across all fractions and runs.

FRACS = [0.10, 0.25, 0.50, 0.75, 1.00]
RUNS  = 3

for run_idx in range(RUNS):
    for frac in FRACS:
        X_sub = X_train_raw.sample(frac=frac, random_state=42)
        y_sub = y_train.loc[X_sub.index]
        n = len(X_sub)

        # fe_train
        t0 = time.perf_counter()
        X_sub_enc, enc_sub = feature_engineering(X_sub, y=y_sub)
        log_row('cpu', frac, n, 'fe_train', run_idx, time.perf_counter() - t0)

        # fe_test  (uses fraction-specific encoders — measures real overhead)
        t0 = time.perf_counter()
        X_test_sub, _ = feature_engineering(X_test_raw, fit_encoders=enc_sub)
        log_row('cpu', frac, n, 'fe_test', run_idx, time.perf_counter() - t0)

        # lr_fit  (Pipeline includes imputer + scaler + LR)
        lr = Pipeline([
            ('imputer',    SimpleImputer(strategy='median')),
            ('scaler',     StandardScaler()),
            ('classifier', LogisticRegression(
                **best_lr_params, random_state=42, max_iter=1000
            ))
        ])
        t0 = time.perf_counter()
        lr.fit(X_sub_enc, y_sub)
        log_row('cpu', frac, n, 'lr_fit', run_idx, time.perf_counter() - t0)

        # lr_predict  (Pipeline applies fitted imputer + scaler + LR)
        t0 = time.perf_counter()
        lr.predict_proba(X_test_sub)
        log_row('cpu', frac, n, 'lr_predict', run_idx, time.perf_counter() - t0)

        # xgb_fit  (Pipeline includes imputer + XGB)
        xgb = Pipeline([
            ('imputer',    SimpleImputer(strategy='median')),
            ('classifier', XGBClassifier(
                **best_xgb_params,
                tree_method='hist', device='cpu',
                random_state=42, eval_metric='logloss'
            ))
        ])
        t0 = time.perf_counter()
        xgb.fit(X_sub_enc, y_sub)
        log_row('cpu', frac, n, 'xgb_fit', run_idx, time.perf_counter() - t0)

        # xgb_predict
        t0 = time.perf_counter()
        xgb.predict_proba(X_test_sub)
        log_row('cpu', frac, n, 'xgb_predict', run_idx, time.perf_counter() - t0)

        print(f'run={run_idx} frac={frac:.0%} n={n:,} done')

    # Save after every outer run — Colab sessions can die mid-sweep
    pd.DataFrame(_rows).to_csv('results/timings_cpu.csv', index=False)
    print(f'[run {run_idx}] Saved results/timings_cpu.csv  ({len(_rows)} rows so far)')


In [ ]:
df_cpu = pd.read_csv('results/timings_cpu.csv')
expected = RUNS * len(FRACS) * 6
print(f'Total rows: {len(df_cpu)}  (expected {expected})')
assert len(df_cpu) == expected, f'Row count mismatch: got {len(df_cpu)}'
print('Row count OK.')
print()
print(df_cpu.groupby(['phase', 'fraction'])['seconds'].median().unstack().to_string())
